# Step 1: Authenticate Earth Engine, import snowMapper, and load configuration settings

In [ ]:
import ee
import geemap
ee.Authenticate(force=False)
import snowmapper as sm

config = sm.read_config('config.yml')     
for key, value in config.items():
    if isinstance(value, dict):
        globals()[key] = value
        for subkey, subval in value.items():
            globals()[subkey] = subval
    else:
        globals()[key] = value
ee.Initialize(project=ee_project)
Map = geemap.Map()

mask = sm.create_mask(masks, domain_ee, CRS, SCALE)

Map.addLayer(domain_bounds_ee, {'color': 'green'}, 'domain')
Map.addLayer(mask, {'min': 0, 'max': 1, 'palette': ['purple', 'yellow']}, 'mask')
Map.addLayer(subdomain_ee.style(color='blue', fillColor='00000000', lineType = 'solid'), {}, 'subdomain')
# Map.addLayer(stations_ee, {'color': 'cyan'}, 'training stations')
Map.centerObject(domain_bounds_ee)
Map

# Step 2 (optional): Monthly multi-year snow cover probabilities

#### Deactivate last four lines **ONLY** if `sc_probab == TRUE` in `decision_tree_settings` and/or `input_vars` (default configuration: `FALSE`)

In [ ]:
# Run the following to check if SC probabilities Image Collection exists & create it if it doesn't
if decision_tree_settings['sc_probab']=='' or 'sc_probab' not in input_vars:
    print("SC probabilities Image Collection is not required - proceeding without")
    pass
else:
    try:
        Map.addLayer(sc_probab_col.first().select('sc_probab'), {'min':0, 'max':1}, 'SC probab (1984-2025)')
        print("SC probabilities Image Collection exists")
    except Exception as e:
        print("Creating SC probabilities Image Collection")
        # multiyear_preprocessed_col = sm.preprocess(domain_bounds_ee, scprobab_start_date, scprobab_end_date, months_list, missions)
        # multiyear_binary_col = sm.binary_snow(multiyear_preprocessed_col, thresholds, active_method)
        # sc_probab_col = sm.sc_probabilities(multiyear_binary_col, months_list, mask)
        # sm.export_ee(sc_probab_col, ee_project, "sc_probab", roi_path, SC_PROBAB_START_YEAR, SC_PROBAB_END_YEAR, domain_bounds_ee, CRS, SCALE, CRS_TRANSFORM)

# Step 3 (optional): Create training dataset & train classifier

In [ ]:
# Run the following to check if pre-trained classifier exists & create it if it doesn't
try:
    classifier
    print("Classifier exists")
except Exception as e:
    print("Classifier doesn't exist")
    processed_stations = sm.process_stations(SC_THRES, stations, input_dir, output_root)
    training_ee = sm.train_dataset(processed_stations, stations_ee, sample_size, CRS, SCALE,
                                   months_list, SC_THRES, output_root, sc_probab_col,
                                   slope, aspect, chili, mtpi, landcover)
    classifier = sm.train_classifier(training_ee, settings, input_vars, active_classifier, ee_project, output_root)

# Step 4: Export preprocessed metadata collection

In [ ]:
preprocessed_col = sm.preprocess(domain_bounds_ee, start_date, end_date, months_list, missions)
composite_col = sm.daily_composites(preprocessed_col)
binary_col = sm.binary_snow(composite_col, thresholds, active_method, domain_ee, SCALE)
synth_col = sm.img_synth(binary_col, domain_bounds_ee, initialisation_date, end_date)
meta_col = sm.add_metadata(synth_col, domain_bounds_ee, constants, input_vars, decision_tree_settings, meteo_format, meteo, roi_path, ee_project, elev, meteo_elev, sc_probab_col, slope, aspect, chili, mtpi, landcover, mask, initialisation_date, end_date, START_HOUR)
meta_col = sm.initial_state(meta_col, domain_bounds_ee, initialisation_date, start_date)
sm.export_ee(meta_col, ee_project, "meta", roi_path, START_YEAR, END_YEAR, domain_bounds_ee, CRS, SCALE, CRS_TRANSFORM)

# Step 5: Export reconstructed collection

In [ ]:
meta_col = ee.ImageCollection(f"projects/{ee_project}/assets/snowMapper/{roi_path}/meta_{roi_path}_{START_YEAR}_{END_YEAR}").sort('system:time_start')
reconstructed_col = sm.reconstruct_daily(meta_col, initialisation_date, start_date, decision_tree_settings, classifier, input_vars, output_vars)
sm.export_ee(reconstructed_col, ee_project, "reconstructed", roi_path, START_YEAR, END_YEAR, domain_bounds_ee, CRS, SCALE, CRS_TRANSFORM)

# Step 6: Export temporal aggregates

In [ ]:
reconstructed_col = ee.ImageCollection(f"projects/{ee_project}/assets/snowMapper/{roi_path}/reconstructed_{roi_path}_{START_YEAR}_{END_YEAR}").sort('system:time_start')
meta_col = ee.ImageCollection(f"projects/{ee_project}/assets/snowMapper/{roi_path}/meta_{roi_path}_{START_YEAR}_{END_YEAR}").sort('system:time_start')
final_col = sm.monthly_aggregates(reconstructed_col, meta_col, months_list, SPIN_UP_PERIOD)
sm.export_ee(final_col, ee_project, "monthly", roi_path, START_YEAR, END_YEAR, domain_bounds_ee, CRS, SCALE, CRS_TRANSFORM)

# Step 7: Export spatial aggregates

In [ ]:
final_col = ee.ImageCollection(f"projects/{ee_project}/assets/snowMapper/{roi_path}/monthly_{roi_path}_{START_YEAR}_{END_YEAR}")
reduced_regions_ee = sm.subdomain_reduction(final_col, subdomain_ee, SCALE, CRS)
task = ee.batch.Export.table.toDrive(
    collection=reduced_regions_ee,
    description=f"snowMapper_{roi_path}_{START_YEAR}-{END_YEAR}_monthly_{SCALE}m_aggr",
    folder='snowMapper',
    fileNamePrefix=f"snowMapper_{roi_path}_{START_YEAR}-{END_YEAR}_monthly_{SCALE}m_aggr",
    fileFormat='CSV'
)
task.start()

# Step 8: Delete old image collections to make space

In [ ]:
sm.rm_ee_asset(
    collection_path = f"projects/{ee_project}/assets/snowMapper/<roi_path>/<collection_name>_<START_YEAR>_<END_YEAR>"  # Example path
)